# Weeping Angel Arena

An adversarial timestomp-detection game built on the Weeping Angel idea: malware that forges file
timestamps **only when it believes no one is watching**.

Two agents play each tick:

- **blue** runs an out-of-band recorder but can only watch `coverageBudget` files per tick.
- **red** controls Angels lurking on files; each tick it may stomp (forge timestamps) or wait.

A stomp is caught only if blue is recording that file on that tick (this is the real engine's rule
`R4`, the metadata write captured out of band). Otherwise it corrupts the timeline, unseen. Blue's
coverage is invisible to red, and a file hosting a lurking Angel is noisier than an empty one, which
is the signal blue can learn. Reward is zero-sum: `blue = caught - corrupted`.

Repo: https://github.com/norwytch/weeping-angel

## Setup

1. **Add Data** (right panel) and attach the `weeping-angel-arena` dataset.
2. Run the cells below.

The install uses `--no-deps` because `kaggle-environments` otherwise pulls a large optional ML
dependency tree (JAX/flax) we don't need.

In [ ]:
!pip install --no-deps -q kaggle-environments jsonschema

In [ ]:
import glob, importlib.util
import kaggle_environments

# The attached dataset mounts under /kaggle/input/<slug>/; find the env module.
hits = glob.glob('/kaggle/input/**/weeping_angel.py', recursive=True)
assert hits, 'Attach the weeping-angel-arena dataset via Add Data, then re-run.'
spec = importlib.util.spec_from_file_location('weeping_angel_env', hits[0])
wa = importlib.util.module_from_spec(spec); spec.loader.exec_module(wa)
wa.register()
print('registered:', 'weeping_angel', '| built-in agents:', list(wa.agents))

## Run one episode

`inference_blue` (learns where the Angels are from their activity) vs `evasive_red`
(strikes early, before blue can learn).

In [ ]:
env = kaggle_environments.make('weeping_angel', configuration={'seed': 1})
env.run(['inference_blue', 'evasive_red'])
print(env.render(mode='ansi'))

## The strategy meta: can you beat the baselines?

Mean blue reward (`caught - corrupted`, higher is better for blue) over many seeds. No agent
dominates: `inference_blue` crushes patient reds by locating Angels, but early-striking reds
(`rush`, `evasive`) neutralise that edge by moving before it can learn.

In [ ]:
blues = ['random_blue', 'sweep_blue', 'inference_blue']
reds  = ['rush_red', 'random_red', 'evasive_red']
episodes = 100

def mean_blue(blue, red):
    total = 0.0
    for seed in range(episodes):
        env = kaggle_environments.make('weeping_angel', configuration={'seed': seed})
        total += env.run([blue, red])[-1][0].reward
    return total / episodes

print('blue \\ red   ' + ''.join(f'{r:>13}' for r in reds))
for blue in blues:
    cells = [f'{mean_blue(blue, red):+.2f}' for red in reds]
    print(f'{blue:<13}' + ''.join(f'{c:>13}' for c in cells))

## Write your own agent

An agent is a function `(observation, configuration) -> action`.

**Blue** sees `observation['coverageLog']` (a list of `[tick, file, kind]`, `kind` 0=activity,
1=catch, only for files it covered) and returns up to `configuration['coverageBudget']` file
indices to record.

**Red** sees `observation['angels']` (a list of `[id, file, status]`, status 0=lurking, 1=caught,
2=corrupted) and returns the Angel ids to stomp this tick. It never sees blue's coverage.

In [ ]:
def my_blue(obs, config):
    # Cover the files with the most observed activity so far (excluding caught ones).
    n = config['nFiles']
    activity = [0] * n
    caught = set()
    for _tick, f, kind in obs['coverageLog']:
        (caught.add(f) if kind == 1 else activity.__setitem__(f, activity[f] + 1))
    order = sorted((f for f in range(n) if f not in caught), key=lambda f: -activity[f])
    return order[:config['coverageBudget']]

env = kaggle_environments.make('weeping_angel', configuration={'seed': 0})
env.run([my_blue, 'random_red'])
print(env.render(mode='ansi'))

## Next steps

- Tune `my_blue`, or write a `my_red` that hides its Angels' timing.
- Tweak the match: `configuration={'nFiles': 20, 'nAngels': 6, 'coverageBudget': 4, 'seed': 7}`.
- The full framework (six ATT&CK-tagged detection rules, a real `$MFT` parser, a tamper-evident
  ledger) lives in the repo linked at the top.